# Embedding Space Visualization

Minh họa không gian embedding **trước** và **sau** contrastive learning (MAMS polonly checkpoint).

- **BEFORE:** DeBERTa pretrained + projection head random (chưa train contrastive)
- **AFTER:** Loaded `outputs_p5_embed_v4/embedding_best.pt` (MAMS polonly, pol_match@5 ≈ 0.822)

Dữ liệu: 973 test records từ `data/processed/classification.jsonl`

In [ ]:
# Cell 1: Setup + load data
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE

import torch
from transformers import AutoTokenizer
from tqdm import tqdm

from src.embedding.model import ContrastiveEmbedder

ROOT = os.path.abspath('..')
DATA_PATH  = os.path.join(ROOT, 'data', 'processed', 'classification.jsonl')
CKPT_PATH  = os.path.join(ROOT, 'kaggle_upload', 'outputs_p5_embed_v4', 'embedding_best.pt')
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LEN    = 128
BATCH_SIZE = 16
DEVICE     = torch.device('cpu')

# --- Colors and markers ---
POL_COLOR  = {'positive': '#2196F3', 'negative': '#F44336', 'neutral': '#757575'}
POL_LABEL  = {'positive': 'Positive', 'negative': 'Negative', 'neutral': 'Neutral'}
CATS = ['food', 'anecdotes/miscellaneous', 'service', 'ambience', 'price']
CAT_MARKER = {'food': 'o', 'anecdotes/miscellaneous': 's', 'service': '^', 'ambience': 'D', 'price': 'P'}
CAT_SHORT  = {'food': 'Food', 'anecdotes/miscellaneous': 'Misc', 'service': 'Service',
              'ambience': 'Ambience', 'price': 'Price'}

# --- Load test records ---
records = []
with open(DATA_PATH) as f:
    for line in f:
        r = json.loads(line)
        if r['split'] == 'test':
            records.append(r)

print(f'Test records: {len(records)}')
sentences  = [r['sentence']         for r in records]
polarities = [r['polarity']         for r in records]
categories = [r['aspect_category']  for r in records]

from collections import Counter
print('Polarity dist:', Counter(polarities))
print('Category dist:', Counter(categories))

In [ ]:
# Cell 2: Encode BEFORE — pretrained DeBERTa + random projection head
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode_sentences(model, sentences, batch_size=BATCH_SIZE, desc='Encoding'):
    """Encode list of sentences → numpy array [N, 256]"""
    model.eval()
    all_vecs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), batch_size), desc=desc):
            batch = sentences[i:i + batch_size]
            enc = tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors='pt'
            )
            vecs = model.encode(
                enc['input_ids'].to(DEVICE),
                enc['attention_mask'].to(DEVICE)
            )
            all_vecs.append(vecs.cpu().numpy())
    return np.concatenate(all_vecs, axis=0)

print('\nBuilding BEFORE model (pretrained DeBERTa + random projection)...')
model_before = ContrastiveEmbedder(
    model_name=MODEL_NAME, proj_dim=256, num_polarities=3
).to(DEVICE)
# NOTE: intentionally NOT loading checkpoint → projection head stays random

vectors_before = encode_sentences(model_before, sentences, desc='BEFORE')
print(f'vectors_before shape: {vectors_before.shape}')
del model_before  # free memory

In [ ]:
# Cell 3: Encode AFTER — trained MAMS polonly checkpoint
print('Building AFTER model (loading checkpoint)...')
model_after = ContrastiveEmbedder(
    model_name=MODEL_NAME, proj_dim=256, num_polarities=3
).to(DEVICE)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True)
model_after.load_state_dict(ckpt)
print(f'Checkpoint loaded from: {CKPT_PATH}')

vectors_after = encode_sentences(model_after, sentences, desc='AFTER')
print(f'vectors_after shape: {vectors_after.shape}')
del model_after

In [ ]:
# Cell 4: Compute metrics
# BEFORE: test-to-test (khong co train index cho random projection model)
# AFTER:  test-to-train (dung train_vectors.npy -- dung voi pipeline thuc te)

TRAIN_VECS_PATH = os.path.join(ROOT, 'kaggle_upload', 'outputs_p5_embed_v4', 'train_vectors.npy')
TRAIN_META_PATH = os.path.join(ROOT, 'kaggle_upload', 'outputs_p5_embed_v4', 'train_metadata.jsonl')

train_vecs = np.load(TRAIN_VECS_PATH)
train_pols, train_cats = [], []
with open(TRAIN_META_PATH) as f:
    for line in f:
        r = json.loads(line)
        train_pols.append(r['polarity'])
        train_cats.append(r['aspect_category'])
print(f'Train vectors: {train_vecs.shape}  ({len(train_pols)} labels)')

def compute_metrics_tt(vecs, labels_pol, labels_cat, k=5):
    """Test-to-test: dung cho BEFORE."""
    N = len(vecs)
    sim = vecs @ vecs.T
    np.fill_diagonal(sim, -2.0)
    mask = np.ones((N, N), bool)
    np.fill_diagonal(mask, False)
    mean_cos = float(sim[mask].mean())
    top_k = np.argsort(-sim, axis=1)[:, :k]
    pol_arr, cat_arr = np.array(labels_pol), np.array(labels_cat)
    pol_m = sum((pol_arr[top_k[i]] == pol_arr[i]).sum() for i in range(N))
    cat_m = sum((cat_arr[top_k[i]] == cat_arr[i]).sum() for i in range(N))
    return {'mean_cosine': mean_cos,
            f'pol_match@{k}': pol_m / (N * k),
            f'cat_match@{k}': cat_m / (N * k),
            'mode': 'test->test'}

def compute_metrics_t2train(test_vecs, test_pols, test_cats,
                             tr_vecs, tr_pols, tr_cats, k=5):
    """Test-to-train: dung cho AFTER -- dung voi cach pipeline hoat dong."""
    sim = test_vecs @ tr_vecs.T
    sim_tt = test_vecs @ test_vecs.T
    np.fill_diagonal(sim_tt, -2.0)
    mask = np.ones((len(test_vecs), len(test_vecs)), bool)
    np.fill_diagonal(mask, False)
    mean_cos = float(sim_tt[mask].mean())
    top_k = np.argsort(-sim, axis=1)[:, :k]
    test_pol, tr_pol = np.array(test_pols), np.array(tr_pols)
    test_cat, tr_cat = np.array(test_cats), np.array(tr_cats)
    N = len(test_vecs)
    pol_m = sum((tr_pol[top_k[i]] == test_pol[i]).sum() for i in range(N))
    cat_m = sum((tr_cat[top_k[i]] == test_cat[i]).sum() for i in range(N))
    return {'mean_cosine': mean_cos,
            f'pol_match@{k}': pol_m / (N * k),
            f'cat_match@{k}': cat_m / (N * k),
            'mode': 'test->train'}

print('Computing BEFORE metrics (test->test)...')
metrics_before = compute_metrics_tt(vectors_before, polarities, categories)
print('BEFORE:', metrics_before)

print('Computing AFTER metrics (test->train)...')
metrics_after = compute_metrics_t2train(
    vectors_after, polarities, categories,
    train_vecs, train_pols, train_cats
)
print('AFTER: ', metrics_after)

In [ ]:
# Cell 5: t-SNE scatter plots -- layout doc (2x1), lon va ro hon
SEED = 42
print('Running t-SNE BEFORE...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb_before = tsne.fit_transform(vectors_before)

print('Running t-SNE AFTER...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb_after = tsne.fit_transform(vectors_after)

fig, axes = plt.subplots(2, 1, figsize=(11, 18))

for ax, emb, title, metrics in [
    (axes[0], emb_before, '(a) Before Contrastive Training', metrics_before),
    (axes[1], emb_after,  '(b) After Contrastive Training',  metrics_after),
]:
    for pol in ['positive', 'negative', 'neutral']:
        for cat in CATS:
            mask = [p == pol and c == cat for p, c in zip(polarities, categories)]
            if not any(mask):
                continue
            idx = np.where(mask)[0]
            ax.scatter(
                emb[idx, 0], emb[idx, 1],
                c=POL_COLOR[pol], marker=CAT_MARKER[cat],
                s=28, alpha=0.70, linewidths=0
            )

    pol_handles = [
        mpatches.Patch(color=POL_COLOR[p], label=POL_LABEL[p])
        for p in ['positive', 'negative', 'neutral']
    ]
    cat_handles = [
        plt.Line2D([0], [0], marker=CAT_MARKER[c], color='gray', linestyle='None',
                   markersize=8, label=CAT_SHORT[c])
        for c in CATS
    ]
    l1 = ax.legend(handles=pol_handles, loc='upper left', title='Polarity',
                   fontsize=10, title_fontsize=10)
    ax.add_artist(l1)
    ax.legend(handles=cat_handles, loc='upper right', title='Category',
              fontsize=10, title_fontsize=10)

    k = 5
    mode = metrics.get('mode', '')
    pol_val = metrics[f'pol_match@{k}']
    cat_val = metrics[f'cat_match@{k}']
    cos_val = metrics['mean_cosine']
    info = (f"pol_match@{k} = {pol_val:.3f}  cat_match@{k} = {cat_val:.3f}\n"
            f"mean cosine = {cos_val:.4f}  ({mode})")
    ax.set_title(title, fontsize=15, fontweight='bold', pad=12)
    ax.text(0.02, 0.02, info, transform=ax.transAxes, fontsize=10,
            verticalalignment='bottom',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.85))
    ax.set_xlabel('t-SNE dim 1', fontsize=11)
    ax.set_ylabel('t-SNE dim 2', fontsize=11)
    ax.tick_params(labelsize=9)

fig.suptitle('Embedding Space: Before vs After Contrastive Learning\n(Test Set, n=973)',
             fontsize=16, fontweight='bold')
plt.tight_layout(h_pad=4)
out = os.path.join(ROOT, 'notebooks', 'fig_tsne_before_after.png')
plt.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved {out}')

In [ ]:
# Cell 6: Cosine similarity histograms — THỰC TẾ (before vs after)
def sample_cosine_similarities(vecs, n_sample=5000, seed=42):
    """Sample n_sample random off-diagonal cosine similarities."""
    rng = np.random.default_rng(seed)
    N = len(vecs)
    pairs = set()
    while len(pairs) < n_sample:
        i = rng.integers(0, N)
        j = rng.integers(0, N)
        if i != j:
            pairs.add((min(i, j), max(i, j)))
    pairs = list(pairs)[:n_sample]
    ii = np.array([p[0] for p in pairs])
    jj = np.array([p[1] for p in pairs])
    sims = np.einsum('ij,ij->i', vecs[ii], vecs[jj])
    return sims

cos_before = sample_cosine_similarities(vectors_before)
cos_after  = sample_cosine_similarities(vectors_after)

fig, ax = plt.subplots(figsize=(8, 5))
bins = np.linspace(-1.0, 1.0, 60)
ax.hist(cos_before, bins=bins, alpha=0.6, color='#FF9800', label='Before contrastive training',
        density=True, edgecolor='none')
ax.hist(cos_after,  bins=bins, alpha=0.6, color='#1976D2', label='After contrastive training',
        density=True, edgecolor='none')

ax.axvline(cos_before.mean(), color='#FF9800', linestyle='--', linewidth=1.5,
           label=f'Before mean = {cos_before.mean():.3f}')
ax.axvline(cos_after.mean(),  color='#1976D2', linestyle='--', linewidth=1.5,
           label=f'After mean  = {cos_after.mean():.3f}')

ax.set_xlabel('Cosine Similarity', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Pairwise Cosine Similarity Distribution\nBefore vs After Contrastive Learning', fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'notebooks', 'fig_cosine_hist.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_cosine_hist.png')

In [ ]:
# Cell 7: Mock / Conceptual scatter plots — phù hợp thesis
rng = np.random.default_rng(0)

# --- Mock data generation ---
# BEFORE: one big cluster, all polarities mixed
n_total = 400
mock_before_xy  = rng.normal(0, 1.8, (n_total, 2))
mock_before_pol = rng.choice(['positive', 'negative', 'neutral'],
                              size=n_total, p=[0.60, 0.25, 0.15])
mock_before_cat = rng.choice(CATS, size=n_total,
                              p=[0.33, 0.31, 0.16, 0.11, 0.09])

# AFTER: 5 category clusters, within each cluster slight polarity sub-structure but overlap
cat_centers = {
    'food':                   np.array([ 0.0,  4.0]),
    'anecdotes/miscellaneous': np.array([ 4.0,  1.0]),
    'service':                np.array([-4.0,  1.0]),
    'ambience':               np.array([ 2.5, -3.5]),
    'price':                  np.array([-2.5, -3.5]),
}
pol_offsets = {'positive': np.array([0.2, 0.2]),
               'negative': np.array([-0.2, 0.1]),
               'neutral':  np.array([0.0, -0.2])}

n_per_cat = {'food': 130, 'anecdotes/miscellaneous': 124, 'service': 63, 'ambience': 47, 'price': 36}

mock_after_xy  = []
mock_after_pol = []
mock_after_cat = []

for cat, n_cat in n_per_cat.items():
    pol_samples = rng.choice(['positive', 'negative', 'neutral'],
                              size=n_cat, p=[0.60, 0.25, 0.15])
    center = cat_centers[cat]
    for pol in pol_samples:
        offset = pol_offsets[pol]
        xy = rng.normal(center + offset, 0.9, (1, 2))
        mock_after_xy.append(xy[0])
        mock_after_pol.append(pol)
        mock_after_cat.append(cat)

mock_after_xy  = np.array(mock_after_xy)
mock_after_pol = np.array(mock_after_pol)
mock_after_cat = np.array(mock_after_cat)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel A — BEFORE
ax = axes[0]
for pol in ['positive', 'negative', 'neutral']:
    for cat in CATS:
        mask = (mock_before_pol == pol) & (mock_before_cat == cat)
        if mask.sum() == 0:
            continue
        ax.scatter(mock_before_xy[mask, 0], mock_before_xy[mask, 1],
                   c=POL_COLOR[pol], marker=CAT_MARKER[cat], s=22, alpha=0.6, linewidths=0)

# Panel B — AFTER
ax2 = axes[1]
for pol in ['positive', 'negative', 'neutral']:
    for cat in CATS:
        mask = (mock_after_pol == pol) & (mock_after_cat == cat)
        if mask.sum() == 0:
            continue
        ax2.scatter(mock_after_xy[mask, 0], mock_after_xy[mask, 1],
                    c=POL_COLOR[pol], marker=CAT_MARKER[cat], s=22, alpha=0.65, linewidths=0)

# Add category labels to panel B
for cat, center in cat_centers.items():
    ax2.annotate(CAT_SHORT[cat], center, fontsize=9, fontweight='bold',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.25', fc='white', alpha=0.7, ec='gray'))

for ax, title in [(axes[0], '(a) Before Contrastive Training'), (axes[1], '(b) After Contrastive Training')]:
    pol_handles = [
        mpatches.Patch(color=POL_COLOR[p], label=POL_LABEL[p])
        for p in ['positive', 'negative', 'neutral']
    ]
    cat_handles = [
        plt.Line2D([0], [0], marker=CAT_MARKER[c], color='gray', linestyle='None',
                   markersize=7, label=CAT_SHORT[c])
        for c in CATS
    ]
    l1 = ax.legend(handles=pol_handles, loc='upper left', title='Polarity',
                   fontsize=8, title_fontsize=8)
    ax.add_artist(l1)
    ax.legend(handles=cat_handles, loc='upper right', title='Category',
              fontsize=8, title_fontsize=8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Dimension 1', fontsize=9)
    ax.set_ylabel('Dimension 2', fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Conceptual Embedding Space: Before vs After Contrastive Learning',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'notebooks', 'fig_tsne_mock.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_tsne_mock.png')

In [ ]:
# Cell 8: Zoom-in on 'food' category cluster — THỰC TẾ
food_idx = [i for i, c in enumerate(categories) if c == 'food']
food_vecs = vectors_after[food_idx]
food_pols = [polarities[i] for i in food_idx]
print(f'Food records: {len(food_idx)}')
print('Polarity dist:', Counter(food_pols))

# t-SNE on food subset
print('Running t-SNE on food subset...')
tsne_food = TSNE(n_components=2, random_state=SEED, perplexity=min(30, len(food_idx) // 4), n_iter=1000)
emb_food = tsne_food.fit_transform(food_vecs)

fig, ax = plt.subplots(figsize=(7, 6))
for pol in ['positive', 'negative', 'neutral']:
    idx = [i for i, p in enumerate(food_pols) if p == pol]
    if not idx:
        continue
    idx = np.array(idx)
    ax.scatter(emb_food[idx, 0], emb_food[idx, 1],
               c=POL_COLOR[pol], label=POL_LABEL[pol],
               s=28, alpha=0.75, edgecolors='none')

# Compute pol_match@5 within food
food_metrics = compute_metrics(food_vecs, food_pols,
                                ['food'] * len(food_pols), k=5)
info = (f"pol_match@5 = {food_metrics['pol_match@5']:.3f}\n"
        f"mean cosine = {food_metrics['mean_cosine']:.4f}")
ax.text(0.02, 0.02, info, transform=ax.transAxes, fontsize=9,
        verticalalignment='bottom',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85))

ax.set_title('Zoom-In: \'Food\' Category Cluster (After Training)\nPolarity Distribution Within Category',
             fontsize=12, fontweight='bold')
ax.set_xlabel('t-SNE dim 1', fontsize=9)
ax.set_ylabel('t-SNE dim 2', fontsize=9)
ax.legend(title='Polarity', fontsize=9, title_fontsize=9)
ax.tick_params(labelsize=7)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'notebooks', 'fig_tsne_food_zoomin.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_tsne_food_zoomin.png')